# Necessary Libraries

In [ ]:
!pip install transformers datasets accelerate scikit-learn einops

# Load & Prepare Data

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load the dataset from Hugging Face
# This dataset has ~3,000+ promoter samples
raw_dataset = load_dataset("neuralbioinfo/bacterial_promoters")

# We want 'segment' (the DNA string) and 'y' (the 0/1 label)
def prepare_data(example):
    return {
        "sequence": example["segment"].upper(),
        "label": int(example["y"])
    }

dataset = raw_dataset['train'].map(prepare_data)

# Split into 80% Train, 20% Test
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_data = split_dataset['train']
test_data = split_dataset['test']

print(f"Dataset Loaded! Training samples: {len(train_data)}, Test samples: {len(test_data)}")
print(f"Sample: {train_data[0]['sequence'][:20]}... Label: {train_data[0]['label']}")

# Load the Teacher Model (DNABERT-2)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Check available GPUs for Kaggle T4x2
num_gpus = torch.cuda.device_count()
print(f"Number of GPUs available: {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the 'no-flashattention' version to ensure compatibility
model_name = "quietflamingo/dnabert2-no-flashattention"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2, trust_remote_code=True
)

# UPDATED: Increased max_length from 100 to 300 for better context capture
def tokenize_func(examples):
    return tokenizer(examples["sequence"], padding="max_length", truncation=True, max_length=300)

tokenized_train = train_data.map(tokenize_func, batched=True)
tokenized_test = test_data.map(tokenize_func, batched=True)

print(f"Model Loaded! Using {num_gpus} GPU(s) for training.")
print(f"Tokenization: max_length=300 (increased from 100 for better context)")

# Train the Teacher (Fine-Tuning with Anti-Overfitting)

## Changes to Fix Overfitting:

| Technique | Original | New | Purpose |
|-----------|----------|-----|---------|
| Learning Rate | 2e-5 | 1e-5 | Slower, more stable learning |
| LR Scheduler | Linear | Cosine | Smooth decay prevents overshooting |
| Warmup | None | 10% | Gradual ramp-up avoids early divergence |
| Weight Decay | 0.01 | 0.05 | Stronger L2 regularization |
| Label Smoothing | 0 | 0.1 | Prevents overconfident predictions |
| Gradient Clipping | None | 1.0 | Prevents exploding gradients |
| Layer Freezing | None | First 10/12 | Only fine-tune top 2 layers |
| Early Stopping | None | 3 epochs | Stop when val loss stops improving |
| Best Model Metric | F1 | Loss | Monitor overfitting directly |
| Data Augmentation | None | Reverse complement + mutations | Increase effective dataset size |
| Epochs | 15 | 10 | Reduced (early stopping will likely trigger earlier) |

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset

# ===== FOCAL LOSS FOR CLASS IMBALANCE =====
class FocalLoss(nn.Module):
    """Focal Loss to handle class imbalance in promoter/non-promoter classification"""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

class FocalLossTrainer(Trainer):
    """Custom Trainer that uses Focal Loss instead of standard Cross-Entropy"""
    def __init__(self, focal_alpha=0.25, focal_gamma=2.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = self.focal_loss(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ===== ENHANCED DNA-SPECIFIC DATA AUGMENTATION =====
def augment_dna_sequence(sequence, aug_prob=0.5):
    """Apply multiple augmentation strategies to DNA sequence"""
    augmentations = []
    complement = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}
    
    # 1. Reverse complement (always valid biologically)
    if random.random() < aug_prob:
        rev_comp = ''.join(complement.get(b, b) for b in reversed(sequence))
        augmentations.append(rev_comp)
    
    # 2. Random point mutations (2% rate, simulates SNPs)
    if random.random() < aug_prob:
        mutated = list(sequence)
        for i in range(len(mutated)):
            if random.random() < 0.02:
                mutated[i] = random.choice(['A', 'T', 'G', 'C'])
        augmentations.append(''.join(mutated))
    
    # 3. Random insertion (1-3 bp, simulates indels)
    if random.random() < aug_prob * 0.5:
        pos = random.randint(0, len(sequence))
        insert = ''.join(random.choices(['A', 'T', 'G', 'C'], k=random.randint(1, 3)))
        augmentations.append(sequence[:pos] + insert + sequence[pos:])
    
    # 4. Random deletion (1-3 bp)
    if random.random() < aug_prob * 0.5:
        if len(sequence) > 10:
            pos = random.randint(0, max(0, len(sequence) - 3))
            del_len = random.randint(1, 3)
            augmentations.append(sequence[:pos] + sequence[pos + del_len:])
    
    return augmentations

def augment_dataset(dataset, augment_prob=0.3):
    """Augment the training dataset with biologically-relevant transformations"""
    augmented_data = {"sequence": [], "label": []}
    
    for example in dataset:
        # Always keep original
        augmented_data["sequence"].append(example["sequence"])
        augmented_data["label"].append(example["label"])
        
        # Add augmentations with probability
        if random.random() < augment_prob:
            augmentations = augment_dna_sequence(example["sequence"])
            for aug_seq in augmentations:
                augmented_data["sequence"].append(aug_seq)
                augmented_data["label"].append(example["label"])
    
    return Dataset.from_dict(augmented_data)

# ===== APPLY AUGMENTATION =====
print("Augmenting training data with enhanced DNA transformations...")
augmented_train = augment_dataset(train_data, augment_prob=0.3)
print(f"Original training size: {len(train_data)}, Augmented size: {len(augmented_train)}")

# Tokenize augmented data
tokenized_train_aug = augmented_train.map(tokenize_func, batched=True)

# ===== LAYER FREEZING (Partial Fine-tuning) =====
print("\nFreezing first 10 encoder layers (keeping last 2 unfrozen)...")
frozen_count = 0
unfrozen_count = 0
for name, param in model.named_parameters():
    if "encoder.layer" in name:
        layer_num = int(name.split("encoder.layer.")[1].split(".")[0])
        if layer_num < 10:  # DNABERT-2 has 12 layers, freeze first 10
            param.requires_grad = False
            frozen_count += 1
        else:
            unfrozen_count += 1
    elif "pooler" in name or "classifier" in name:
        unfrozen_count += 1
    else:
        param.requires_grad = False
        frozen_count += 1

print(f"Frozen parameters: {frozen_count}, Unfrozen parameters: {unfrozen_count}")

def compute_metrics(pred):
    labels = pred.label_ids
    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

# ===== UPDATED TRAINING CONFIGURATION =====
# Key changes for reaching 85%+ accuracy:
# 1. Focal Loss for class imbalance
# 2. Reduced batch size (16) for longer sequences (max_length=300)
# 3. Increased gradient accumulation (4) to maintain effective batch size
# 4. All anti-overfitting measures from before

training_args = TrainingArguments(
    output_dir="/kaggle/working/teacher_results",
    eval_strategy="epoch",
    save_strategy="epoch",

    # Learning rate optimization
    learning_rate=1e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    
    # Training duration
    num_train_epochs=10,
    
    # Regularization
    weight_decay=0.05,
    label_smoothing_factor=0.1,
    max_grad_norm=1.0,

    # UPDATED: Reduced batch size for longer sequences (max_length=300)
    per_device_train_batch_size=16,  # Reduced from 32
    per_device_eval_batch_size=16,   # Reduced from 32
    gradient_accumulation_steps=4,    # Increased from 2 to maintain effective batch size
    eval_accumulation_steps=4,
    
    # Mixed precision for faster training on T4
    fp16=True,
    
    # Multi-GPU settings
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    ddp_find_unused_parameters=False,

    # Early stopping configuration
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    logging_steps=50,
    report_to="none",
    remove_unused_columns=True,
    save_total_limit=2,
)

# ===== CREATE TRAINER WITH FOCAL LOSS =====
trainer = FocalLossTrainer(
    focal_alpha=0.25,
    focal_gamma=2.0,
    model=model,
    args=training_args,
    train_dataset=tokenized_train_aug,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("\n" + "="*60)
print("TRAINING CONFIGURATION (Phase 1: Quick Wins)")
print("="*60)
print(f"Loss function: Focal Loss (alpha=0.25, gamma=2.0)")
print(f"Max sequence length: 300 (increased from 100)")
print(f"Learning rate: {training_args.learning_rate}")
print(f"LR Scheduler: {training_args.lr_scheduler_type} with {training_args.warmup_ratio*100}% warmup")
print(f"Weight decay: {training_args.weight_decay}")
print(f"Label smoothing: {training_args.label_smoothing_factor}")
print(f"Per-device batch size: {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * torch.cuda.device_count() * training_args.gradient_accumulation_steps}")
print(f"Early stopping patience: 3 epochs")
print("="*60)

print("\nStarting training with Focal Loss + Longer Sequences...")
trainer.train()

# Evaluation

In [ ]:
# ===== EVALUATION ON ALL TEST SETS =====
import time

# 1. Final Evaluation on main test set
print("="*60)
print("EVALUATION RESULTS")
print("="*60)

eval_results = trainer.evaluate()
print("\n1. Main Test Set (20% of train split):")
print(f"   Accuracy: {eval_results['eval_accuracy']:.2%}")
print(f"   F1 Score: {eval_results['eval_f1']:.4f}")
print(f"   Loss: {eval_results['eval_loss']:.4f}")

# 2. Evaluate on Sigma-70 test set
print("\n2. Loading and evaluating on Sigma-70 test set...")
test_sigma70 = raw_dataset['test_sigma70'].map(prepare_data)
tokenized_sigma70 = test_sigma70.map(tokenize_func, batched=True)
sigma70_results = trainer.evaluate(tokenized_sigma70)
print(f"   Sigma-70 Accuracy: {sigma70_results['eval_accuracy']:.2%}")
print(f"   Sigma-70 F1 Score: {sigma70_results['eval_f1']:.4f}")
print(f"   Sigma-70 Loss: {sigma70_results['eval_loss']:.4f}")

# 3. Evaluate on Multi-species test set
print("\n3. Loading and evaluating on Multi-species test set...")
test_multispecies = raw_dataset['test_multispecies'].map(prepare_data)
tokenized_multispecies = test_multispecies.map(tokenize_func, batched=True)
multispecies_results = trainer.evaluate(tokenized_multispecies)
print(f"   Multi-species Accuracy: {multispecies_results['eval_accuracy']:.2%}")
print(f"   Multi-species F1 Score: {multispecies_results['eval_f1']:.4f}")
print(f"   Multi-species Loss: {multispecies_results['eval_loss']:.4f}")

# 4. Summary table
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"{'Test Set':<25} {'Accuracy':<12} {'F1 Score':<12} {'Loss':<10}")
print("-"*60)
print(f"{'Main (20% train)':<25} {eval_results['eval_accuracy']:<12.2%} {eval_results['eval_f1']:<12.4f} {eval_results['eval_loss']:<10.4f}")
print(f"{'Sigma-70':<25} {sigma70_results['eval_accuracy']:<12.2%} {sigma70_results['eval_f1']:<12.4f} {sigma70_results['eval_loss']:<10.4f}")
print(f"{'Multi-species':<25} {multispecies_results['eval_accuracy']:<12.2%} {multispecies_results['eval_f1']:<12.4f} {multispecies_results['eval_loss']:<10.4f}")
print("="*60)

# 5. Benchmark Inference Speed
print("\nBenchmarking inference speed...")
eval_model = trainer.model
if hasattr(eval_model, 'module'):
    eval_model = eval_model.module
eval_model = eval_model.to(device)
eval_model.eval()

sample_inputs = tokenizer(["ACGTAGCT"*10]*100, return_tensors="pt", padding=True, max_length=300, truncation=True).to(device)

with torch.no_grad():
    _ = eval_model(**sample_inputs)
torch.cuda.synchronize()

start_time = time.time()
with torch.no_grad():
    _ = eval_model(**sample_inputs)
torch.cuda.synchronize()
end_time = time.time()

print(f"Time to process 100 sequences: {end_time - start_time:.4f} seconds")

# 6. Save the model
save_path = "/kaggle/working/final_teacher_model"
eval_model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"\nModel saved to {save_path}")

# 7. Check if target accuracy reached
target_accuracy = 0.85
if eval_results['eval_accuracy'] >= target_accuracy:
    print(f"\n✓ TARGET REACHED: {eval_results['eval_accuracy']:.2%} >= {target_accuracy:.0%}")
else:
    print(f"\n→ Current: {eval_results['eval_accuracy']:.2%}, Target: {target_accuracy:.0%}")
    print("  Consider implementing Phase 2 (K-Fold Ensemble) for further improvement.")

# Phase 2: K-Fold Ensemble (Optional)

Run this cell if Phase 1 accuracy is below 85%. K-Fold cross-validation with ensemble averaging can provide an additional 3-5% improvement.

**Expected improvement:** 82-84% accuracy with 5-fold ensemble

In [ ]:
# ===== PHASE 2: K-FOLD ENSEMBLE TRAINING =====
# Only run this if Phase 1 accuracy is below 85%

from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader
import copy

def train_kfold_ensemble(dataset, n_splits=5):
    """Train multiple models using K-Fold cross-validation for ensemble"""
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    # Get labels for stratification
    labels = [ex['label'] for ex in dataset]
    
    models = []
    fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(range(len(dataset)), labels)):
        print(f"\n{'='*60}")
        print(f"Training Fold {fold + 1}/{n_splits}")
        print(f"{'='*60}")
        
        # Create fold datasets
        train_fold = dataset.select(train_idx)
        val_fold = dataset.select(val_idx)
        
        # Apply augmentation to training fold
        augmented_train_fold = augment_dataset(train_fold, augment_prob=0.3)
        
        # Tokenize
        tokenized_train_fold = augmented_train_fold.map(tokenize_func, batched=True)
        tokenized_val_fold = val_fold.map(tokenize_func, batched=True)
        
        # Fresh model for each fold
        fold_model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=2, trust_remote_code=True
        )
        
        # Apply layer freezing
        for name, param in fold_model.named_parameters():
            if "encoder.layer" in name:
                layer_num = int(name.split("encoder.layer.")[1].split(".")[0])
                if layer_num < 10:
                    param.requires_grad = False
        
        # Training args for each fold
        fold_training_args = TrainingArguments(
            output_dir=f"/kaggle/working/fold_{fold}_results",
            eval_strategy="epoch",
            save_strategy="epoch",
            learning_rate=1e-5,
            warmup_ratio=0.1,
            lr_scheduler_type="cosine",
            num_train_epochs=10,
            weight_decay=0.05,
            label_smoothing_factor=0.1,
            max_grad_norm=1.0,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            gradient_accumulation_steps=4,
            fp16=True,
            dataloader_num_workers=4,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            logging_steps=100,
            report_to="none",
            save_total_limit=1,
        )
        
        # Train fold
        fold_trainer = FocalLossTrainer(
            focal_alpha=0.25,
            focal_gamma=2.0,
            model=fold_model,
            args=fold_training_args,
            train_dataset=tokenized_train_fold,
            eval_dataset=tokenized_val_fold,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
        )
        
        fold_trainer.train()
        
        # Evaluate and save
        results = fold_trainer.evaluate()
        fold_scores.append(results['eval_accuracy'])
        models.append(fold_model)
        
        print(f"Fold {fold + 1} Accuracy: {results['eval_accuracy']:.2%}")
        print(f"Fold {fold + 1} F1: {results['eval_f1']:.4f}")
    
    print(f"\n{'='*60}")
    print(f"K-FOLD CROSS-VALIDATION RESULTS")
    print(f"{'='*60}")
    print(f"Mean CV Accuracy: {np.mean(fold_scores):.2%} (+/- {np.std(fold_scores):.2%})")
    return models, fold_scores

def ensemble_predict(models, tokenized_dataset):
    """Make predictions using ensemble of models (average logits)"""
    all_probs = []
    
    for i, fold_model in enumerate(models):
        print(f"Getting predictions from model {i+1}/{len(models)}...")
        fold_model.eval()
        fold_model.to(device)
        
        # Create a simple dataloader
        fold_model_probs = []
        batch_size = 32
        
        for start_idx in range(0, len(tokenized_dataset), batch_size):
            end_idx = min(start_idx + batch_size, len(tokenized_dataset))
            batch = tokenized_dataset[start_idx:end_idx]
            
            input_ids = torch.tensor(batch['input_ids']).to(device)
            attention_mask = torch.tensor(batch['attention_mask']).to(device)
            
            with torch.no_grad():
                outputs = fold_model(input_ids=input_ids, attention_mask=attention_mask)
                probs = F.softmax(outputs.logits, dim=-1)
                fold_model_probs.append(probs.cpu())
        
        all_probs.append(torch.cat(fold_model_probs))
    
    # Average probabilities across models
    ensemble_probs = torch.stack(all_probs).mean(dim=0)
    predictions = ensemble_probs.argmax(dim=-1).numpy()
    
    return predictions, ensemble_probs.numpy()

# Train the ensemble
print("Starting K-Fold Ensemble Training...")
print("This will train 5 models - expect longer training time.\n")

ensemble_models, cv_scores = train_kfold_ensemble(train_data, n_splits=5)

# Ensemble evaluation on test set
print("\nEvaluating ensemble on test sets...")
test_labels = np.array([ex['label'] for ex in test_data])
ensemble_preds, ensemble_probs = ensemble_predict(ensemble_models, tokenized_test)

ensemble_accuracy = accuracy_score(test_labels, ensemble_preds)
ensemble_f1 = f1_score(test_labels, ensemble_preds)

print(f"\n{'='*60}")
print("ENSEMBLE RESULTS")
print(f"{'='*60}")
print(f"Main Test Set Ensemble Accuracy: {ensemble_accuracy:.2%}")
print(f"Main Test Set Ensemble F1: {ensemble_f1:.4f}")

# Compare with single model
print(f"\nImprovement over single model:")
print(f"  Single model accuracy: {eval_results['eval_accuracy']:.2%}")
print(f"  Ensemble accuracy:     {ensemble_accuracy:.2%}")
print(f"  Improvement:           {(ensemble_accuracy - eval_results['eval_accuracy'])*100:+.2f}%")

# Evaluate ensemble on additional test sets
print("\nEvaluating ensemble on Sigma-70 test set...")
sigma70_labels = np.array([ex['label'] for ex in test_sigma70])
sigma70_ensemble_preds, _ = ensemble_predict(ensemble_models, tokenized_sigma70)
sigma70_ensemble_acc = accuracy_score(sigma70_labels, sigma70_ensemble_preds)
sigma70_ensemble_f1 = f1_score(sigma70_labels, sigma70_ensemble_preds)
print(f"Sigma-70 Ensemble Accuracy: {sigma70_ensemble_acc:.2%}")
print(f"Sigma-70 Ensemble F1: {sigma70_ensemble_f1:.4f}")

print("\nEvaluating ensemble on Multi-species test set...")
multispecies_labels = np.array([ex['label'] for ex in test_multispecies])
multispecies_ensemble_preds, _ = ensemble_predict(ensemble_models, tokenized_multispecies)
multispecies_ensemble_acc = accuracy_score(multispecies_labels, multispecies_ensemble_preds)
multispecies_ensemble_f1 = f1_score(multispecies_labels, multispecies_ensemble_preds)
print(f"Multi-species Ensemble Accuracy: {multispecies_ensemble_acc:.2%}")
print(f"Multi-species Ensemble F1: {multispecies_ensemble_f1:.4f}")

# Final summary
print(f"\n{'='*60}")
print("FINAL ENSEMBLE SUMMARY")
print(f"{'='*60}")
print(f"{'Test Set':<25} {'Accuracy':<12} {'F1 Score':<12}")
print("-"*60)
print(f"{'Main (20% train)':<25} {ensemble_accuracy:<12.2%} {ensemble_f1:<12.4f}")
print(f"{'Sigma-70':<25} {sigma70_ensemble_acc:<12.2%} {sigma70_ensemble_f1:<12.4f}")
print(f"{'Multi-species':<25} {multispecies_ensemble_acc:<12.2%} {multispecies_ensemble_f1:<12.4f}")
print("="*60)

if ensemble_accuracy >= 0.85:
    print(f"\n✓ TARGET REACHED: {ensemble_accuracy:.2%} >= 85%")
else:
    print(f"\n→ Current: {ensemble_accuracy:.2%}, Target: 85%")
    print("  Consider additional techniques from Phase 3 or hyperparameter tuning.")